In [4]:
from datasets import load_dataset

# Carga el dataset de Hugging Face
dataset = load_dataset("SINAI/ALIA-es-discriminative-stance-detection")

# Inspeccionar las particiones (train, validation, test)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'target', 'description', 'comment', 'annotation_1', 'annotation_2', 'annotation_3', 'majority_label'],
        num_rows: 3000
    })
})


In [5]:
import pandas as pd

# Convertimos el dataset a DataFrame
df = dataset["train"].to_pandas()

# Cantidad de comentarios por clase
print(df["majority_label"].value_counts())

majority_label
favor      1780
against     627
neutral     443
            150
Name: count, dtype: int64


In [6]:
# Comentarios sin etiqueta mayoritaria
sin_etiqueta = df[df["majority_label"] == ""]

print("Cantidad:", len(sin_etiqueta))

# Ver las anotaciones de los primeros 10
print(sin_etiqueta[
    ["annotation_1", "annotation_2", "annotation_3"]
].head(10))

Cantidad: 150
    annotation_1 annotation_2 annotation_3
2        against      neutral        favor
27       neutral        favor      against
29         favor      neutral      against
45       against        favor      neutral
58       against      neutral        favor
102        favor      against      neutral
111        favor      neutral      against
154      neutral      against        favor
161      against        favor      neutral
166        favor      against      neutral


In [7]:
# Verificar si los 150 casos tienen tres anotaciones distintas
sin_etiqueta[
    ["annotation_1", "annotation_2", "annotation_3"]
].nunique(axis=1).value_counts()

3    150
Name: count, dtype: int64

In [8]:
# Excluir comentarios sin etiqueta mayoritaria
df_limpio = df[df["majority_label"] != ""].copy()

# Verificar cantidad de comentarios
print("Comentarios originales:", len(df))
print("Comentarios utilizables:", len(df_limpio))

# Ver distribución de clases
print(df_limpio["majority_label"].value_counts())

Comentarios originales: 3000
Comentarios utilizables: 2850
majority_label
favor      1780
against     627
neutral     443
Name: count, dtype: int64


In [9]:
# Contar cuántas etiquetas distintas hay por comentario
acuerdos = df[
    ["annotation_1", "annotation_2", "annotation_3"]
].nunique(axis=1)

# Calcular tipos de acuerdo
acuerdo_total = (acuerdos == 1).sum()
acuerdo_parcial = (acuerdos == 2).sum()
sin_acuerdo = (acuerdos == 3).sum()

print("Acuerdo total:", acuerdo_total)
print("Acuerdo parcial:", acuerdo_parcial)
print("Sin acuerdo:", sin_acuerdo)

# Porcentajes
print("\nPorcentajes:")
print("Acuerdo total:", round(acuerdo_total / len(df) * 100, 2), "%")
print("Acuerdo parcial:", round(acuerdo_parcial / len(df) * 100, 2), "%")
print("Sin acuerdo:", round(sin_acuerdo / len(df) * 100, 2), "%")

Acuerdo total: 1539
Acuerdo parcial: 1311
Sin acuerdo: 150

Porcentajes:
Acuerdo total: 51.3 %
Acuerdo parcial: 43.7 %
Sin acuerdo: 5.0 %


In [10]:
# Calcular cantidad de palabras por comentario
df_limpio["num_palabras"] = df_limpio["comment"].str.split().str.len()

# Mostrar estadísticas
print(df_limpio["num_palabras"].describe())

count    2850.000000
mean       54.138596
std        32.073506
min         7.000000
25%        28.000000
50%        46.000000
75%        76.000000
max       155.000000
Name: num_palabras, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

# Primera división: 70% entrenamiento, 30% restante
train_df, temp_df = train_test_split(
    df_limpio,
    test_size=0.30,
    random_state=42,
    stratify=df_limpio["majority_label"]
)

# Segunda división: mitad validación, mitad test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["majority_label"]
)

print("Entrenamiento:", len(train_df))
print("Validación:", len(val_df))
print("Test:", len(test_df))

Entrenamiento: 1995
Validación: 427
Test: 428


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score

# Crear el modelo
modelo = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clasificador", LogisticRegression(max_iter=1000))
])

# Entrenar con los 1995 comentarios
modelo.fit(
    train_df["comment"],
    train_df["majority_label"]
)

# Predecir los 427 comentarios de validación
predicciones = modelo.predict(val_df["comment"])

# Evaluar resultados
print("Macro-F1:", f1_score(
    val_df["majority_label"],
    predicciones,
    average="macro"
))

print("Accuracy:", accuracy_score(
    val_df["majority_label"],
    predicciones
))

Macro-F1: 0.2969769846909843
Accuracy: 0.6229508196721312


In [13]:
from sklearn.metrics import classification_report

print(classification_report(
    val_df["majority_label"],
    predicciones,
    zero_division=0
))

              precision    recall  f1-score   support

     against       0.28      0.05      0.09        94
       favor       0.64      0.97      0.77       267
     neutral       0.33      0.02      0.03        66

    accuracy                           0.62       427
   macro avg       0.42      0.35      0.30       427
weighted avg       0.51      0.62      0.51       427



In [14]:
# Modelo con pesos balanceados
modelo_balanceado = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Entrenar
modelo_balanceado.fit(
    train_df["comment"],
    train_df["majority_label"]
)

# Predecir en validación
pred_balanceadas = modelo_balanceado.predict(val_df["comment"])

# Evaluar
print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_balanceadas,
    average="macro"
))

print("\nResultados por clase:")
print(classification_report(
    val_df["majority_label"],
    pred_balanceadas,
    zero_division=0
))

Macro-F1: 0.4522187604433166

Resultados por clase:
              precision    recall  f1-score   support

     against       0.43      0.54      0.48        94
       favor       0.72      0.69      0.71       267
     neutral       0.19      0.15      0.17        66

    accuracy                           0.58       427
   macro avg       0.45      0.46      0.45       427
weighted avg       0.58      0.58      0.57       427



In [15]:
modelo_bigramas = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_bigramas.fit(
    train_df["comment"],
    train_df["majority_label"]
)

pred_bigramas = modelo_bigramas.predict(val_df["comment"])

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_bigramas,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_bigramas,
    zero_division=0
))

Macro-F1: 0.47769261473175256
              precision    recall  f1-score   support

     against       0.41      0.49      0.45        94
       favor       0.73      0.80      0.76       267
     neutral       0.42      0.15      0.22        66

    accuracy                           0.63       427
   macro avg       0.52      0.48      0.48       427
weighted avg       0.61      0.63      0.61       427



In [16]:
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

for c in [0.01, 0.1, 1, 10, 100]:

    modelo_prueba = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clasificador", LogisticRegression(
            C=c,
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    modelo_prueba.fit(train_df["comment"], train_df["majority_label"])

    pred = modelo_prueba.predict(val_df["comment"])
    
    f1 = f1_score(
        val_df["majority_label"],
        pred,
        average="macro"
    )

    print(f"C={c}: Macro-F1={f1:.4f}")

C=0.01: Macro-F1=0.4571
C=0.1: Macro-F1=0.4512
C=1: Macro-F1=0.4777
C=10: Macro-F1=0.4449
C=100: Macro-F1=0.4525


In [17]:
modelo_caracteres = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5)
    )),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_caracteres.fit(
    train_df["comment"],
    train_df["majority_label"]
)

pred_caracteres = modelo_caracteres.predict(val_df["comment"])

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_caracteres,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_caracteres,
    zero_division=0
))

Macro-F1: 0.4357141316771258
              precision    recall  f1-score   support

     against       0.41      0.48      0.44        94
       favor       0.71      0.71      0.71       267
     neutral       0.18      0.14      0.16        66

    accuracy                           0.57       427
   macro avg       0.43      0.44      0.44       427
weighted avg       0.56      0.57      0.57       427



In [18]:
# Combinar tema y comentario
X_train = train_df["target"].fillna("") + " " + train_df["comment"].fillna("")
X_val = val_df["target"].fillna("") + " " + val_df["comment"].fillna("")

modelo_con_target = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_con_target.fit(X_train, train_df["majority_label"])

pred_target = modelo_con_target.predict(X_val)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_target,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_target,
    zero_division=0
))

Macro-F1: 0.49239509559350697
              precision    recall  f1-score   support

     against       0.46      0.56      0.51        94
       favor       0.74      0.78      0.76       267
     neutral       0.32      0.15      0.21        66

    accuracy                           0.64       427
   macro avg       0.51      0.50      0.49       427
weighted avg       0.62      0.64      0.62       427



In [19]:
X_train_completo = (
    train_df["target"].fillna("") + " " +
    train_df["description"].fillna("") + " " +
    train_df["comment"].fillna("")
)

X_val_completo = (
    val_df["target"].fillna("") + " " +
    val_df["description"].fillna("") + " " +
    val_df["comment"].fillna("")
)

modelo_completo = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_completo.fit(
    X_train_completo,
    train_df["majority_label"]
)

pred_completo = modelo_completo.predict(X_val_completo)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_completo,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_completo,
    zero_division=0
))

Macro-F1: 0.4714662819501529
              precision    recall  f1-score   support

     against       0.43      0.53      0.48        94
       favor       0.74      0.63      0.68       267
     neutral       0.23      0.29      0.26        66

    accuracy                           0.56       427
   macro avg       0.47      0.48      0.47       427
weighted avg       0.59      0.56      0.57       427



In [20]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report

modelo_svm = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LinearSVC(
        C=1,
        class_weight="balanced",
        random_state=42
    ))
])

modelo_svm.fit(X_train, train_df["majority_label"])

pred_svm = modelo_svm.predict(X_val)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_svm,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_svm,
    zero_division=0
))

Macro-F1: 0.4613757353316921
              precision    recall  f1-score   support

     against       0.51      0.45      0.48        94
       favor       0.72      0.88      0.79       267
     neutral       0.26      0.08      0.12        66

    accuracy                           0.66       427
   macro avg       0.50      0.47      0.46       427
weighted avg       0.60      0.66      0.62       427



In [21]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score

configuraciones = ParameterGrid({
    "min_df": [1, 2, 3, 5],
    "sublinear_tf": [True, False],
    "max_features": [None, 5000, 10000]
})

resultados = []

for config in configuraciones:
    modelo_busqueda = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            **config
        )),
        ("clasificador", LogisticRegression(
            C=1,
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    modelo_busqueda.fit(X_train, train_df["majority_label"])
    pred = modelo_busqueda.predict(X_val)

    f1 = f1_score(
        val_df["majority_label"],
        pred,
        average="macro"
    )

    resultados.append({
        **config,
        "Macro-F1": f1
    })

resultados_df = pd.DataFrame(resultados)

display(
    resultados_df.sort_values(
        "Macro-F1",
        ascending=False
    ).head(10)
)

,max_features,min_df,sublinear_tf,Macro-F1
3,NaN,2,False,0.517155
5,NaN,3,False,0.507369
21,10000.0,3,False,0.507369
19,10000.0,2,False,0.504509
17,10000.0,1,False,0.504429
4,NaN,3,True,0.500488
20,10000.0,3,True,0.500184
2,NaN,2,True,0.498684
18,10000.0,2,True,0.495974
11,5000.0,2,False,0.495871


In [22]:
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

modelo_optimizado = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=False,
        max_features=None
    )),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Entrenamiento
inicio = time.perf_counter()
modelo_optimizado.fit(X_train, train_df["majority_label"])
tiempo_entrenamiento_opt = time.perf_counter() - inicio

# Inferencia
inicio = time.perf_counter()
pred_optimizado = modelo_optimizado.predict(X_val)
tiempo_inferencia_opt = time.perf_counter() - inicio

# Métricas
y_real = val_df["majority_label"]

print("Accuracy:", accuracy_score(y_real, pred_optimizado))
print("Precision Macro:", precision_score(y_real, pred_optimizado, average="macro"))
print("Recall Macro:", recall_score(y_real, pred_optimizado, average="macro"))
print("Macro-F1:", f1_score(y_real, pred_optimizado, average="macro"))
print("Weighted-F1:", f1_score(y_real, pred_optimizado, average="weighted"))

print("\nReporte completo:")
print(classification_report(y_real, pred_optimizado, zero_division=0))

print("\nMatriz de confusión:")
print(confusion_matrix(
    y_real,
    pred_optimizado,
    labels=["against", "favor", "neutral"]
))

print("\nTiempos:")
print("Entrenamiento:", round(tiempo_entrenamiento_opt, 3), "segundos")
print("Inferencia:", round(tiempo_inferencia_opt, 3), "segundos")
print("Por comentario:", round(tiempo_inferencia_opt / len(X_val) * 1000, 3), "ms")

Accuracy: 0.6323185011709602
Precision Macro: 0.524874664767735
Recall Macro: 0.5188944187270746
Macro-F1: 0.5171550795585741
Weighted-F1: 0.6275807605900855

Reporte completo:
              precision    recall  f1-score   support

     against       0.45      0.54      0.49        94
       favor       0.75      0.76      0.76       267
     neutral       0.37      0.26      0.30        66

    accuracy                           0.63       427
   macro avg       0.52      0.52      0.52       427
weighted avg       0.63      0.63      0.63       427


Matriz de confusión:
[[ 51  33  10]
 [ 46 202  19]
 [ 16  33  17]]

Tiempos:
Entrenamiento: 0.392 segundos
Inferencia: 0.028 segundos
Por comentario: 0.066 ms


In [23]:
resumen_modelos = pd.DataFrame([
    ["Baseline", 0.2970],
    ["Balanceado", 0.4522],
    ["Bigramas", 0.4777],
    ["Bigramas + target", 0.4924],
    ["SVM", 0.4614],
    ["TF-IDF optimizado", 0.5172]
], columns=["Modelo", "Macro-F1 validación"])

display(resumen_modelos)

,Modelo,Macro-F1 validación
0,Baseline,0.2970
1,Balanceado,0.4522
2,Bigramas,0.4777
3,Bigramas + target,0.4924
4,SVM,0.4614
5,TF-IDF optimizado,0.5172


In [24]:
import joblib

joblib.dump(modelo_optimizado, "modelo_clasico.pkl")

['modelo_clasico.pkl']

## BETO sin fine-tuning

In [25]:
%pip install "numpy<2" "torch==2.2.2" "transformers==4.44.2" "accelerate==0.34.2"

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install "pandas==2.2.3" "datasets==3.6.0"

Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip install "pyarrow==17.0.0"

Note: you may need to restart the kernel to use updated packages.


In [28]:
import torch
import transformers
import datasets
import pandas
import numpy

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("GPU disponible:", torch.cuda.is_available())

PyTorch: 2.2.2
Transformers: 4.44.2
Datasets: 3.6.0
Pandas: 2.2.3
NumPy: 1.26.4
GPU disponible: False


In [29]:
from transformers import AutoTokenizer, AutoModel
import torch

nombre_modelo = "dccuchile/bert-base-spanish-wwm-cased"

# Cargar tokenizador y modelo preentrenado
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
beto = AutoModel.from_pretrained(nombre_modelo)

# Usar CPU y modo evaluación
device = torch.device("cpu")
beto.to(device)
beto.eval()

print("BETO cargado correctamente")

/Users/paulaho/opt/anaconda3/envs/tp3_nlp/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BETO cargado correctamente


In [30]:
textos_train = (
    train_df["target"].fillna("").astype(str)
    + " [SEP] " +
    train_df["comment"].fillna("").astype(str)
).tolist()

textos_val = (
    val_df["target"].fillna("").astype(str)
    + " [SEP] " +
    val_df["comment"].fillna("").astype(str)
).tolist()

print("Entrenamiento:", len(textos_train))
print("Validación:", len(textos_val))
print("Ejemplo:", textos_train[0][:200])

Entrenamiento: 1995
Validación: 427
Ejemplo: Ruido nocturno de bares, discotecas y gente en la calle [SEP]  Vivo en un barrio donde el botellón es indiscriminado, zona de Guzmán el Bueno-General Rodrigo- Julián Romea-General Dávila, todas las no
